# Stage 4 final ensemble tuning

Run this notebook after the three classifier notebooks, both risk-score notebooks, and
`models/stage3a_base_classifiers/train_base_models.py`. It compares the final Logistic Regression
meta-model using the Linear Regression risk score and the Decision Tree Regressor risk score.
The input is the nine leak-free classifier probabilities plus one risk score. It saves every
final-model CV result, the selected risk source, the final pipeline, and test metrics.

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, recall_score, make_scorer
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
import sys
sys.path.insert(0, str(Path.cwd() / 'hyper parameter')); sys.path.insert(0, str(Path.cwd()))
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'dataset').exists(): ROOT = ROOT.parent
TRAIN_PATH = ROOT / 'dataset' / 'processed' / 'meta_train.csv'
TEST_PATH = ROOT / 'dataset' / 'processed' / 'meta_test.csv'
RESULTS = ROOT / 'hyper parameter' / 'results' / 'meta_ensemble'
BEST = ROOT / 'hyper parameter' / 'best'
MODEL_DIR = ROOT / 'models' / 'saved_models'
RESULTS.mkdir(parents=True, exist_ok=True); BEST.mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(parents=True, exist_ok=True)
SEED = 19
TARGET = 'casualty_severity'
GROUP = 'collision_index'
BASE_FEATURES = ['logistic_regression_fatal', 'logistic_regression_serious', 'logistic_regression_slight', 'knn_fatal', 'knn_serious', 'knn_slight', 'decision_tree_fatal', 'decision_tree_serious', 'decision_tree_slight']
RISK_FEATURES = ['risk_score_linear_regression', 'risk_score_decision_tree_regressor']

In [2]:
train = pd.read_csv(TRAIN_PATH, dtype={GROUP: str})
test = pd.read_csv(TEST_PATH, dtype={GROUP: str})
required = BASE_FEATURES + RISK_FEATURES + [TARGET, GROUP]
missing = sorted(set(required) - set(train.columns))
if missing: raise ValueError(f'Missing OOF/meta columns: {missing}. Run train_base_models.py first.')
y = train[TARGET].astype(int)
groups = train[GROUP].astype(str)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_splits = list(cv.split(train, y, groups))
global_rates = y.value_counts(normalize=True).reindex([1, 2, 3], fill_value=0.0)
for fold, (_, validation_index) in enumerate(cv_splits):
    rates = y.iloc[validation_index].value_counts(normalize=True).reindex([1, 2, 3], fill_value=0.0)
    if (rates - global_rates).abs().max() > 0.001: raise ValueError(f'Class-rate drift in fold {fold}')
print('meta train:', train.shape, 'meta test:', test.shape)
print('OOF fold count:', len(cv_splits))

meta train: (102305, 13) meta test: (25578, 13)
OOF fold count: 5


In [3]:
PARAM_GRID = {
    'classifier__C': np.logspace(-3, 2, 8),
    'classifier__class_weight': [None, 'balanced', {1: 3, 2: 1.5, 3: 1}, {1: 5, 2: 2, 3: 1}],
    'classifier__solver': ['lbfgs'],
}
scoring = {
    'accuracy': 'accuracy', 'balanced_accuracy': 'balanced_accuracy', 'macro_f1': 'f1_macro',
    'fatal_recall': make_scorer(recall_score, labels=[1], average='macro', zero_division=0),
    'serious_recall': make_scorer(recall_score, labels=[2], average='macro', zero_division=0),
    'slight_recall': make_scorer(recall_score, labels=[3], average='macro', zero_division=0),
}
candidate_summaries = []
candidate_models = {}
for risk_feature in RISK_FEATURES:
    features = BASE_FEATURES + [risk_feature]
    pipeline = Pipeline([('scale', StandardScaler()), ('classifier', LogisticRegression(max_iter=2500, random_state=SEED))])
    search = RandomizedSearchCV(pipeline, PARAM_GRID, scoring=scoring, refit='macro_f1', n_iter=12, random_state=SEED, cv=cv_splits, n_jobs=1, verbose=2, return_train_score=True, error_score='raise')
    search.fit(train[features], y, groups=groups)
    result_table = pd.DataFrame(search.cv_results_).sort_values('rank_test_macro_f1')
    safe_name = risk_feature.replace('risk_score_', '')
    result_table.to_csv(RESULTS / f'{safe_name}_cv_results.csv', index=False)
    candidate_models[risk_feature] = search.best_estimator_
    candidate_summaries.append({'risk_feature': risk_feature, 'features': features, 'best_params': search.best_params_, 'best_cv_macro_f1': float(search.best_score_)})
    print(risk_feature, search.best_params_, search.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END classifier__C=0.13894954943731375, classifier__class_weight=balanced, classifier__solver=lbfgs; total time=   0.1s
[CV] END classifier__C=0.13894954943731375, classifier__class_weight=balanced, classifier__solver=lbfgs; total time=   0.1s
[CV] END classifier__C=0.13894954943731375, classifier__class_weight=balanced, classifier__solver=lbfgs; total time=   0.1s
[CV] END classifier__C=0.13894954943731375, classifier__class_weight=balanced, classifier__solver=lbfgs; total time=   0.1s
[CV] END classifier__C=0.13894954943731375, classifier__class_weight=balanced, classifier__solver=lbfgs; total time=   0.1s
[CV] END classifier__C=0.001, classifier__class_weight={1: 5, 2: 2, 3: 1}, classifier__solver=lbfgs; total time=   0.2s
[CV] END classifier__C=0.001, classifier__class_weight={1: 5, 2: 2, 3: 1}, classifier__solver=lbfgs; total time=   0.2s
[CV] END classifier__C=0.001, classifier__class_weight={1: 5, 2: 2, 3: 1}, clas

In [4]:
summary_frame = pd.DataFrame(candidate_summaries).sort_values('best_cv_macro_f1', ascending=False)
summary_frame.to_csv(RESULTS / 'risk_source_comparison.csv', index=False)
selected_risk = summary_frame.iloc[0]['risk_feature']
selected_features = BASE_FEATURES + [selected_risk]
selected_model = candidate_models[selected_risk]
test_prediction = selected_model.predict(test[selected_features])
metrics = {
    'selected_risk_feature': selected_risk, 'selected_features': selected_features,
    'candidates': candidate_summaries,
    'test_accuracy': float(accuracy_score(test[TARGET], test_prediction)),
    'test_balanced_accuracy': float(balanced_accuracy_score(test[TARGET], test_prediction)),
    'test_macro_f1': float(f1_score(test[TARGET], test_prediction, average='macro', zero_division=0)),
    'test_fatal_recall': float(recall_score(test[TARGET], test_prediction, labels=[1], average=None, zero_division=0)[0]),
    'test_serious_recall': float(recall_score(test[TARGET], test_prediction, labels=[2], average=None, zero_division=0)[0]),
    'test_slight_recall': float(recall_score(test[TARGET], test_prediction, labels=[3], average=None, zero_division=0)[0]),
    'test_confusion_matrix': confusion_matrix(test[TARGET], test_prediction, labels=[1, 2, 3]).tolist(),
}
(RESULTS / 'final_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(MODEL_DIR / 'meta_contract.json').write_text(json.dumps({'features': selected_features, 'risk_feature': selected_risk, 'class_labels': [1, 2, 3]}, indent=2), encoding='utf-8')
pd.DataFrame({'collision_index': test[GROUP], 'actual': test[TARGET], 'predicted': test_prediction}).to_csv(RESULTS / 'test_predictions.csv', index=False)
joblib.dump(selected_model, BEST / 'meta_ensemble_best.pkl')
joblib.dump(selected_model, MODEL_DIR / 'meta_ensemble_3.pkl')
print(json.dumps(metrics, indent=2))

{
  "selected_risk_feature": "risk_score_decision_tree_regressor",
  "selected_features": [
    "logistic_regression_fatal",
    "logistic_regression_serious",
    "logistic_regression_slight",
    "knn_fatal",
    "knn_serious",
    "knn_slight",
    "decision_tree_fatal",
    "decision_tree_serious",
    "decision_tree_slight",
    "risk_score_decision_tree_regressor"
  ],
  "candidates": [
    {
      "risk_feature": "risk_score_linear_regression",
      "features": [
        "logistic_regression_fatal",
        "logistic_regression_serious",
        "logistic_regression_slight",
        "knn_fatal",
        "knn_serious",
        "knn_slight",
        "decision_tree_fatal",
        "decision_tree_serious",
        "decision_tree_slight",
        "risk_score_linear_regression"
      ],
      "best_params": {
        "classifier__solver": "lbfgs",
        "classifier__class_weight": {
          "1": 5,
          "2": 2,
          "3": 1
        },
        "classifier__C": 0.026826957